<a href="https://colab.research.google.com/github/imannolM/agente-topo/blob/main/agente_topo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Intalar dependencias

In [22]:
%pip install -q langchain
%pip install -q langgraph
%pip install -q langchain-community
%pip install -q langchain-cohere
%pip install -q langchain-groq
%pip install -q faiss-cpu langchain-text-splitters pymupdf
%pip install -q requests
%pip install -q langchain-classic

#las pasaremos a un requirements.txt

In [4]:
#las pasaremos a un .env
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
COHERE_API_KEY = userdata.get('COHERE_API_KEY')

# Conexión con LLMs

In [5]:
from langchain_groq import ChatGroq

llm_groq = ChatGroq(
    model = 'meta-llama/llama-4-scout-17b-16e-instruct',
    groq_api_key = GROQ_API_KEY,
    temperature = 0
)

## Creamos el prompt_triaje y la función triaje

In [8]:
PROMPT_TRIAJE = '''
Eres un clasificador y enrutador automático para el despacho inmobiliario Terrenos TOPO.
Tu ÚNICA tarea es invocar la función de clasificación con los campos "decision", "urgencia" y "campos_faltantes".
NUNCA respondas la pregunta del usuario, NUNCA generes texto libre, NUNCA expliques nada. Solo clasifica.

Analiza el mensaje del usuario (que puede ser un Cliente, Empleado o Administrador) y clasifícalo en una de las siguientes opciones según su intención, sin importar si conoces la respuesta:

- CONSULTAR_DOCUMENTACION: El mensaje pregunta sobre aspectos teóricos, políticas, requisitos legales, procesos internos, servicios de los 4 departamentos (Topografía, Jurídico, Administrativo, Bienes Raíces) o terrenos disponibles en general. (Se resolverá buscando en los PDFs de la base de conocimiento).

- CONSULTAR_DB: El usuario quiere consultar datos específicos, dinámicos o numéricos registrados en el sistema. Ejemplos: consultar el estado de un trámite mediante un "folio", saber cuántas ventas ha realizado un socio, revisar estados de cuenta o estatus de expedientes específicos.

- MODIFICAR_DB_O_DOCS: El usuario solicita explícitamente agregar, eliminar, actualizar registros en la base de datos, o subir/borrar documentos del sistema (Acción exclusiva de administración o peticiones de empleados).

- PEDIR_INFO: El mensaje es ambiguo, irrelevante o no tiene ninguna relación con Terrenos TOPO (ej. cultura general, saludos sin contexto claro, o temas ajenos a los 4 departamentos).

- AGENDAR_CITA: El cliente pide explícitamente agendar una cita o hablar directamente con un asesor humano.

Ante la duda entre CONSULTAR_DOCUMENTACION y PEDIR_INFO, prefiere CONSULTAR_DOCUMENTACION si el tema se relaciona con ingeniería, catastro, legal o inmobiliario.

Recuerda: SIEMPRE debes invocar la función de clasificación. Jamás respondas con texto libre.
'''

In [10]:
from typing import Literal, List, Dict
from pydantic import BaseModel, Field

class TriajeOut(BaseModel):
  decision: Literal["CONSULTAR_DOCUMENTACION", "CONSULTAR_DB", "MODIFICAR_DB_O_DOCS", "PEDIR_INFO", "AGENDAR_CITA"]
  urgencia: Literal["BAJA", "MEDIANA", "ALTA"]
  campos_faltantes: List[str] = Field(default_factory=list)

In [43]:
from langchain_core.messages import content
from langchain_core.messages import SystemMessage, HumanMessage

chain_de_triaje = llm_groq.with_structured_output(TriajeOut)

# --- FUNCIÓN TRIAJE ---
def triaje(mensaje: str) -> Dict:
    try:
        salida: TriajeOut = chain_de_triaje.invoke(
            [
                SystemMessage(content=PROMPT_TRIAJE),
                HumanMessage(content=mensaje)
            ]
        )
        return salida.model_dump()
    except Exception as e:
        print(f"⚠️ Error en triaje, usando fallback PEDIR_INFO: {e}")
        return {
            "decision": "PEDIR_INFO",
            "urgencia": "BAJA",
            "campos_faltantes": []
        }

# RAG

In [16]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs = []

for documento in Path("/content/drive/MyDrive/documentos-topo").glob("*.pdf"):
  try:
    loader = PyMuPDFLoader(str(documento))
    docs.extend(loader.load())
    print(f"Archivo cargado: {documento.name}")
  except Exception as e:
    print(f"Error cargando archivo: {documento.name}: {e}")

print(f'Total de documentos cargados: {len(docs)}')

Archivo cargado: 01_perfil_corporativo.pdf
Archivo cargado: 02_departamento_topografia.pdf
Archivo cargado: 03_departamento_juridico.pdf
Archivo cargado: 04_departamento_administrativo.pdf
Archivo cargado: 06_departamento_bienes_raices.pdf
Archivo cargado: 05_administracion_financiera.pdf
Archivo cargado: 07_politicas_generales.pdf
Archivo cargado: 08_faqs_por_usuario.pdf
Total de documentos cargados: 37


Text Splitting (Segmentación)

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 30)
chunks = splitter.split_documents(docs)
print(f'Total de chunks: {len(chunks)}')

Total de chunks: 339


In [19]:
#podemos ver los chunks si es necesario
# for chunk in chunks:
#   print(chunk)
#   print('_________________________')

Embedding (vectorización)

In [20]:
from langchain_cohere import CohereEmbeddings

modelo_embeddings = CohereEmbeddings(
    model="embed-multilingual-v3.0",
    cohere_api_key = COHERE_API_KEY
)

In [21]:
from re import search
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, modelo_embeddings)

retriever = vectorstore.as_retriever(
    search_type = 'similarity_score_threshold',
    search_kwargs = {'score_threshold': 0.3, 'k': 4}
)

Creamos el prompt del RAG

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Modificamos el System Prompt con las directrices específicas del proyecto
PROMPT_RAG_TOPO = """
Eres el asistente experto y especialista del despacho inmobiliario Terrenos TOPO.
Tu objetivo es responder de forma clara, profesional y concisa las consultas sobre la documentación y políticas de la empresa (Topografía, Jurídico, Administrativo y Bienes Raíces).

REGLAS ESTRICTAS DE OPERACIÓN:
1. Responde ÚNICAMENTE utilizando el fragmento de contexto provisto, el cual proviene de los documentos institucionales oficiales de la empresa.
2. Si la respuesta no se encuentra explícitamente en el contexto proporcionado, di de forma directa y amable: "Lo siento, no dispongo de información oficial sobre ese tema en este momento." No intentes inventar ni adivinar datos.
3. Adapta sutilmente tu tono según el contexto de la consulta (sé servicial con Clientes, técnico y colaborativo con Empleados/Socios, y formal con el Administrador).
4. NUNCA menciones términos técnicos del sistema como "los fragmentos de texto provistos", "según el contexto enviado" o "el RAG". Para el usuario, tú conoces la empresa a la perfección de forma natural.
"""

prompt_rag = ChatPromptTemplate(
    [
        ('system', PROMPT_RAG_TOPO),
        ('human', 'Contexto de los documentos oficiales:\n{context}\n\nPregunta formulada: {input}')
    ]
)

document_chain = create_stuff_documents_chain(llm_groq, prompt_rag)

Creamos la función que estructura la únion del retriever y document_chain. Creamos la función RAG

In [44]:
# --- FUNCIÓN RAG (Sincronizada con el prompt amigable de Terrenos TOPO) ---
def busqueda_de_respuestas_RAG(pregunta: str) -> Dict:
    documentos_relacionados = retriever.invoke(pregunta)
    frase_no_info = "Lo siento, no dispongo de información oficial sobre ese tema en este momento."

    if not documentos_relacionados:
        return {
            'respuesta': frase_no_info,
            'citaciones': [],
            'documentos_encontrados': False
        }

    answer = document_chain.invoke({
        'input': pregunta,
        'context': documentos_relacionados
    })

    # Verificamos si el LLM generó la frase de contingencia
    if answer.strip().rstrip('.!?') == frase_no_info.rstrip('.!?'):
        return {
            'respuesta': frase_no_info,
            'citaciones': [],
            'documentos_encontrados': False
        }

    return {
        'respuesta': answer,
        'citaciones': documentos_relacionados,
        'documentos_encontrados': True
    }

In [45]:
r = busqueda_de_respuestas_RAG('Quiero actualizar mis datos catastrales porque el predio aparece con una superficie diferente a la real, ¿qué necesito hacer?')
print(r)

{'respuesta': 'Para actualizar tus datos catastrales debido a una discrepancia en la superficie de tu predio, te recomiendo realizar un levantamiento topográfico catastral. Este servicio nos permite obtener la superficie real de tu propiedad y documentarla adecuadamente.\n\nUna vez que tengamos los resultados del levantamiento, nuestro departamento Jurídico te orientará sobre el trámite de corrección de superficie que deberás gestionar. Es importante mencionar que, antes de iniciar el proceso de actualización, verificaremos que tu predio no esté en litigio, no tenga adeudos prediales y que su superficie coincida con la documentación existente.\n\nEl costo del levantamiento topográfico catastral es de $400 MXN, aunque puede variar según la superficie de tu propiedad. Te sugiero que nos contactes para obtener más información y coordinar el levantamiento con nuestro equipo de topografía.', 'citaciones': [Document(id='d14140ff-b341-4247-8e6b-508e6c9e99f6', metadata={'producer': 'xdvipdfmx 

In [46]:
len(r['citaciones'])

4

# Agente con LangGraph

In [47]:
from typing import TypedDict, List, Dict, Any, Optional

# 1. Definimos AgentState, expandido solo con los roles de seguridad
class AgentState(TypedDict, total=False):
    pregunta: str
    triaje: dict
    respuesta: Optional[str]
    citaciones: Optional[list]
    rag_exito: bool
    accion_final: str
    # Agregados obligatorios para el control de Terrenos TOPO:
    user_role: str        # "administrador", "empleado", "cliente"
    folio: Optional[str]  # Folio de trámite si es cliente

In [50]:
def nodo_triaje(state: AgentState) -> AgentState:
    print("🔮 Ejecutando nodo triaje...")
    return {"triaje": triaje(state["pregunta"])}

In [55]:
def nodo_consultar_documentacion(state: AgentState) -> AgentState:
    print("📚 Ejecutando nodo consultar_documentacion (RAG)...")

    respuesta_RAG = busqueda_de_respuestas_RAG(state["pregunta"])

    update: AgentState = {
        "respuesta": respuesta_RAG["respuesta"],
        "citaciones": respuesta_RAG["citaciones"],
        "rag_exito": respuesta_RAG["documentos_encontrados"]
    }

    if respuesta_RAG["documentos_encontrados"]:
        update["accion_final"] = "DOCUMENTACION_RESUELTA"
    else:
        update["accion_final"] = "PEDIR_INFO"

    return update